# Phase 6 — RAG: Conversational Recommendations
## Fase 6 — RAG: Recomendaciones Conversacionales


**EN:** Builds the conversational engine. The user asks questions in natural language  
and GPT-4o responds using their own notes and the book catalog as context.

**ES:** Construye el motor conversacional. El usuario hace preguntas en lenguaje natural  
y GPT-4o responde usando sus propias notas y el catálogo como contexto.

**RAG Architecture / Arquitectura RAG:**
```
User question → Vectorize (BERT) → Retrieve (notes + catalog) → Prompt → GPT-4o → Response
```


## Step 1 — Install Dependencies / Instalar Dependencias

In [1]:
!pip install openai -q
!pip install transformers -q
!pip install torch -q
print("Dependencies installed.")

Dependencies installed.


## Step 2 — Imports and Mount Drive / Imports y Montar Drive

In [2]:
import logging
import os
import pickle

import numpy as np
import pandas as pd
import torch
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.preprocessing import normalize
from transformers import AutoTokenizer, AutoModel
from openai import OpenAI
from google.colab import drive

logging.basicConfig(level=logging.INFO, format="%(levelname)s:%(name)s:%(message)s")
logger = logging.getLogger("phase_6")

drive.mount('/content/drive')
DRIVE_PATH = "/content/drive/MyDrive/Colab Notebooks/PF_Ironhack/"
logger.info("Imports complete. Drive mounted.")

Mounted at /content/drive


## Step 3 — Configure OpenAI API Key / Configurar API Key

**EN:** Loaded from Colab Secrets — never hardcoded.  
Add it: left sidebar → key icon → secret named `OPENAI_API_KEY`.

**ES:** Cargada desde Colab Secrets — nunca en el código.  
Añadirla: panel izquierdo → icono de llave → secret llamado `OPENAI_API_KEY`.


In [3]:
from google.colab import userdata

api_key = userdata.get('OPENAI_API_KEY')
client  = OpenAI(api_key=api_key)

try:
    client.models.list()
    print("API key loaded and connected.")
    logger.info("OpenAI API key verified.")
except Exception as e:
    print(f"API key error: {e}")
    print("Check that the secret name is exactly: OPENAI_API_KEY")


API key loaded and connected.


## Step 4 — Load All Required Data / Cargar Todos los Datos Necesarios

In [5]:
def load_pickle(filename: str, drive_path: str) -> object:
    """Loads a pickle file from Drive. Raises FileNotFoundError if not found."""
    filepath = os.path.join(drive_path, filename)
    if not os.path.exists(filepath):
        raise FileNotFoundError(
            f"{filename} not found at {filepath}."
        )
    with open(filepath, 'rb') as f:
        obj = pickle.load(f)
    logger.info("Loaded: %s", filename)
    return obj


bert_embeddings    = normalize(load_pickle('bert_embeddings.pkl',   DRIVE_PATH), norm='l2')
catalog_embeddings = load_pickle('catalog_embeddings.pkl', DRIVE_PATH)
reader_profile     = load_pickle('reader_profile.pkl',     DRIVE_PATH)

df_notas   = pd.read_csv(os.path.join(DRIVE_PATH, 'user_notes_clustered.csv'))
df_catalog = pd.read_csv(os.path.join(DRIVE_PATH, 'books_clean.csv'))

print(f"User notes      : {len(df_notas)}")
print(f"Catalog books   : {len(df_catalog):,}")
print(f"Profile clusters: {len(reader_profile)}")


User notes      : 50
Catalog books   : 78,811
Profile clusters: 5


## Step 5 — Load BERT Model / Cargar Modelo BERT

**EN:** Same model as previous phases — vectors must live in the same mathematical space.  
**ES:** Mismo modelo que en fases anteriores — los vectores deben estar en el mismo espacio.


In [6]:
MODEL_NAME = "sentence-transformers/all-mpnet-base-v2"

tokenizer  = AutoTokenizer.from_pretrained(MODEL_NAME)
bert_model = AutoModel.from_pretrained(MODEL_NAME)
bert_model.eval()

logger.info("BERT model ready: %s", MODEL_NAME)
print(f"Model ready: {MODEL_NAME}")


def embed_query(text: str) -> np.ndarray:
    """Converts a user question into a normalized 768-dim BERT embedding.
    Uses mean pooling — same process as Phase 3.
    Args: text — user question in natural language
    Returns: numpy array shape (384,), L2-normalized
    """
    inputs = tokenizer(text, padding=True, truncation=True,
                       max_length=512, return_tensors='pt')
    with torch.no_grad():
        outputs = bert_model(**inputs)
    token_emb = outputs.last_hidden_state
    mask      = inputs['attention_mask']
    mask_exp  = mask.unsqueeze(-1).expand(token_emb.size()).float()
    embedding = (token_emb * mask_exp).sum(1) / mask_exp.sum(1).clamp(min=1e-9)
    return normalize(embedding.squeeze().numpy().reshape(1, -1), norm='l2')[0]


config.json:   0%|          | 0.00/571 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/363 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/438M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

MPNetModel LOAD REPORT from: sentence-transformers/all-mpnet-base-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Model ready: sentence-transformers/all-mpnet-base-v2


## Step 6 — Retriever

**EN:** Finds the most relevant notes and catalog books using cosine similarity.  
These fragments become the context GPT-4o uses — making responses truly personalized.

**ES:** Encuentra las notas y libros más relevantes usando similitud coseno.  
Estos fragmentos se convierten en el contexto que usa GPT-4o.


In [23]:
def retrieve_context(query: str, top_k_notes: int = 3, top_k_books: int = 5) -> dict:
    """Retrieves the most relevant notes and catalog books for a query.
    Args:
        query: user question
        top_k_notes: number of notes to retrieve
        top_k_books: number of catalog books to retrieve
    Returns: dict with keys 'notes' and 'books'
    """
    query_vec = embed_query(query)

    note_sims    = cosine_similarity(query_vec.reshape(1, -1), bert_embeddings)[0]
    top_note_idx = note_sims.argsort()[::-1][:top_k_notes]

    retrieved_notes = []
    for idx in top_note_idx:
        row = df_notas.iloc[idx]
        retrieved_notes.append({
            'title':      row['title'],
            'author':     row['author'],
            'note_text':  row['note_text'],
            'rating':     row['rating'],
            'tags':       row.get('tags', ''),
            'similarity': round(float(note_sims[idx]), 4)
        })

    titles_read  = set(df_notas['title'].str.lower().str.strip().tolist())
    catalog_sims = cosine_similarity(query_vec.reshape(1, -1), catalog_embeddings)[0]
    top_cat_idx  = [
        i for i in catalog_sims.argsort()[::-1]
        if str(df_catalog.iloc[i]['title']).lower().strip() not in titles_read
    ][:top_k_books]

    retrieved_books = []
    for idx in top_cat_idx:
        row = df_catalog.iloc[idx]
        retrieved_books.append({
            'title':       row['title'],
            'author_names':     row['author_names'],
            'description': str(row['description'])[:300],
            'similarity':  round(float(catalog_sims[idx]), 4)
        })

    return {'notes': retrieved_notes, 'books': retrieved_books}


# Quick test
print("Retriever test: 'books about loneliness and philosophy'")
print("-" * 55)
test_ctx = retrieve_context('books about loneliness and philosophy')
for n in test_ctx['notes']:
    print(f"  note  [{n['similarity']}] {n['title']}")
for b in test_ctx['books']:
    print(f"  book  [{b['similarity']}] {b['title']}")


Retriever test: 'books about loneliness and philosophy'
-------------------------------------------------------
  note  [0.3851] The Little Prince
  note  [0.3468] Let the Right One In
  note  [0.3401] Man's Search for Meaning
  book  [0.5993] Eleven Kinds of Loneliness
  book  [0.5721] A History of Loneliness
  book  [0.5481] Party of One: The Loners' Manifesto
  book  [0.5326] The Lonely City: Adventures in the Art of Being Alone
  book  [0.5212] The Opposite of Loneliness: Essays and Stories


## Step 7 — Prompt Builder / Constructor del Prompt

**EN:** Assembles (system prompt + reader profile + retrieved context + question) for GPT-4o.  
**ES:** Construye (system prompt + perfil lector + contexto recuperado + pregunta) para GPT-4o.


In [24]:
def build_prompt(query: str, context: dict, profile: dict) -> tuple:
    """Builds (system_prompt, user_prompt) for the GPT-4o API call."""

    system_prompt = (
        "You are a personal literary assistant with deep knowledge "
        "of the user's reading history and preferences.\n\n"
        "Your role:\n"
        "- Answer questions about the user's notes and reading patterns\n"
        "- Recommend books from the catalog that match the user's profile\n"
        "- Help rediscover forgotten ideas from past readings\n\n"
        "Rules:\n"
        "- Base answers ONLY on the context provided\n"
        "- Be warm, specific, reference the user's actual notes\n"
        "- When recommending, always explain WHY it matches the profile\n"
        "- Keep responses concise but insightful\n"
        f"- IMPORTANT: Always respond in English, regardless of the language of the question."
    )

    profile_summary = "USER'S READING PROFILE:\n"
    for key, cl in profile.items():
        profile_summary += (
            f"- Cluster {cl['cluster_id']}: "
            f"Books: {', '.join(cl['books'])} | "
            f"Themes: {', '.join(cl['top_words'][:4])} | "
            f"Avg rating: {cl['avg_rating']}/5\n"
        )

    notes_ctx = "\nRELEVANT PERSONAL NOTES:\n"
    for n in context['notes']:
        notes_ctx += (
            f"Book: '{n['title']}' by {n['author']} "
            f"(Rating: {n['rating']}/5)\n"
            f"Note: {n['note_text']}\n\n"
        )

    books_ctx = "\nRELEVANT CATALOG BOOKS:\n"
    for b in context['books']:
        books_ctx += (
            f"Title: '{b['title']}' by {b['author_names']}\n"
            f"Description: {b['description']}\n\n"
        )

    user_prompt = profile_summary + notes_ctx + books_ctx + f"\nUSER QUESTION: {query}"
    return system_prompt, user_prompt


print("build_prompt ready.")


build_prompt ready.


## Step 8 — Main RAG Function / Función Principal
```
question → retrieve_context() → build_prompt() → GPT-4o → response
```


In [25]:
def ask(query: str, top_k_notes: int = 3, top_k_books: int = 5, verbose: bool = False) -> str:
    """Main RAG interface: natural language question → personalized response.
    Args:
        query: question in English
        top_k_notes: notes to include as context
        top_k_books: catalog books to include as context
        verbose: if True, prints retrieved context
    Returns: GPT-4o response string
    """
    context = retrieve_context(query, top_k_notes, top_k_books)

    if verbose:
        print('Retrieved context:')
        print(f"  Notes : {[n['title'] for n in context['notes']]}")
        print(f"  Books : {[b['title'] for b in context['books']]}")
        print()

    system_prompt, user_prompt = build_prompt(query, context, reader_profile)

    response = client.chat.completions.create(
        model='gpt-4o',
        messages=[
            {'role': 'system', 'content': system_prompt},
            {'role': 'user',   'content': user_prompt}
        ],
        temperature=0.7,
        max_tokens=600
    )
    return response.choices[0].message.content


print("ask() ready.")


ask() ready.


## Step 9 — Test the RAG / Probar el RAG

**EN:** Four queries covering the main use cases. Each GPT-4o call takes ~3-8 seconds.  
**ES:** Cuatro consultas que cubren los principales casos de uso. Cada llamada tarda ~3-8 segundos.


In [26]:
print("=" * 65)
print("TEST 1 — Recurring themes in reading notes")
print("=" * 65)
print(ask(
    'What recurring themes and emotions appear most in my reading notes?',
    verbose=True
))


TEST 1 — Recurring themes in reading notes
Retrieved context:
  Notes : ['The Little Prince', 'The Royal Game', 'Dune']
  Books : ['Characters, Emotion & Viewpoint: Techniques and Exercises for Crafting Dynamic Characters and Effective Viewpoints', 'How to Read Literature Like a Professor: A Lively and Entertaining Guide to Reading Between the Lines', 'How to Read Literature Like a Professor: A Lively and Entertaining Guide to Reading Between the Lines', 'Falling in Love with Close Reading: Lessons for Analyzing Texts--And Life', 'How to Read Literature']

From your reading notes, the most recurring themes and emotions are:

1. **Loneliness and Connection**: As seen in your note for "The Little Prince," the protagonist's loneliness and the theme of seeing with the heart resonate deeply with you. This suggests a recurring interest in stories that explore the depths of human connection and the feeling of isolation.

2. **Contrasting Characters and Trauma**: In "The Royal Game," you're fa

In [27]:
print("=" * 65)
print("TEST 2 — Recommendation by specific theme")
print("=" * 65)
print(ask(
    'Recommend me books about existentialism and the search for meaning, '
    'similar to what I already enjoy.',
    verbose=True
))


TEST 2 — Recommendation by specific theme
Retrieved context:
  Notes : ['Nausea', 'Siddhartha', "Man's Search for Meaning"]
  Books : ['Irrational Man: A Study in Existential Philosophy', 'Books for Living', 'Existentialism Is a Humanism', 'Prometheus Rising', 'The Novel Cure: From Abandonment to Zestlessness: 751 Books to Cure What Ails You']

Given your appreciation for philosophical exploration and existential themes, especially as exhibited in books like "Nausea" by Jean-Paul Sartre and "Man's Search for Meaning" by Viktor E. Frankl, I recommend the following:

1. **"Irrational Man: A Study in Existential Philosophy" by William Barrett** - This book is a foundational text that introduces existentialism to the English-speaking audience. It provides context and clarity on existential themes, much like what you appreciated in Sartre's vivid depiction of ontology. This book will deepen your understanding of existential philosophy and its impact on literature.

2. **"Existentialism Is a

In [28]:
print("=" * 65)
print("TEST 3 — Rediscovering forgotten ideas")
print("=" * 65)
print(ask(
    'What are the most interesting ideas I had in my past readings '
    'that I might have forgotten?',
    verbose=True
))


TEST 3 — Rediscovering forgotten ideas
Retrieved context:
  Notes : ['Dune', 'The Little Prince', 'Sapiens']
  Books : ['Lit!: A Christian Guide to Reading Books', 'Books for Living', 'Ten Years in the Tub: A Decade Soaking in Great Books', 'The Intellectual Devotional: Revive Your Mind, Complete Your Education, and Roam Confidently with the Cultured Class', 'How to Read Literature Like a Professor: A Lively and Entertaining Guide to Reading Between the Lines']

You have explored some profound ideas across your past readings that resonate with your diverse interests. Here are a few intriguing concepts you might have forgotten:

1. **Interconnectedness in Human History** ('Sapiens' by Yuval Noah Harari): You were fascinated by the idea that collective fictions and shared myths are what bind societies together. This perspective shifted your understanding of human history and the power of storytelling.

2. **Ecology and Religion in Politics** ('Dune' by Frank Herbert): The intricate blend

## Step 10 — Production Module / Módulo de Producción

**EN:** The production version of this RAG pipeline lives in `rag_engine.py` in the project folder.  
That file encapsulates all the logic developed in this notebook into the `RAGEngine` class,  
which is imported directly by the Streamlit app in Phase 7.

Any changes to the RAG logic should be made directly in `rag_engine.py` —  
not regenerated from this notebook.

**ES:** La versión de producción de este pipeline RAG vive en `rag_engine.py` en la carpeta del proyecto.  
Ese archivo encapsula toda la lógica desarrollada en este notebook en la clase `RAGEngine`,  
que es importada directamente por la app Streamlit en la Fase 7.

Cualquier cambio en la lógica RAG debe hacerse directamente en `rag_engine.py` —  
no regenerarse desde este notebook.

| File | Purpose |
|------|---------|
| `Fase_6_RAG.ipynb` | Development, testing, and documentation |
| `rag_engine.py` | Production module — imported by Streamlit |